In [1]:
import pandas as pd
import numpy as np
import os

# Chemins des dossiers
PROCESSED_PATH = "../data/processed/"
FINAL_PATH = "../data/final/"

# Créer le dossier final s'il n'existe pas
os.makedirs(FINAL_PATH, exist_ok=True)

# Chargement du dataset nettoyé au Notebook 02
df = pd.read_csv(os.path.join(PROCESSED_PATH, "dataset_preprocessed.csv"))

print(f"✅ Données chargées pour le Feature Engineering : {df.shape}")

✅ Données chargées pour le Feature Engineering : (5423, 51)


In [3]:
# --- Étape de vérification des noms ---
print("Colonnes disponibles :", df.columns.tolist())

# On définit les noms réels (à corriger ici si tu vois des noms différents dans la liste au-dessus)
# Si tu vois 'age' au lieu de 'AGE_YRS', change-le ici :
COL_AGE = 'AGE_YRS' if 'AGE_YRS' in df.columns else 'age'
COL_ALLERGIES = 'ALLERGIES' if 'ALLERGIES' in df.columns else 'allergies'
COL_HISTORY = 'HISTORY' if 'HISTORY' in df.columns else 'medical_history'

def calculate_vulnerability(row):
    score = 0
    
    # 1. Vérification de l'âge
    age = row.get(COL_AGE, 30) # 30 par défaut si non trouvé
    if age > 65 or age < 5:
        score += 2
        
    # 2. Vérification des allergies
    allergies = str(row.get(COL_ALLERGIES, '')).lower()
    if allergies not in ['unknown', 'none', '', 'nan']:
        score += 3
        
    # 3. Vérification des antécédents
    history = str(row.get(COL_HISTORY, '')).lower()
    if history not in ['unknown', 'none', '', 'nan']:
        score += 2
        
    return score

# Application sécurisée
df['vulnerability_score'] = df.apply(calculate_vulnerability, axis=1)

# Création d'une feature de risque binaire simple
df['is_high_risk'] = (df['vulnerability_score'] >= 5).astype(int)

print(f"✅ Feature Engineering terminé sur {len(df)} lignes.")
print(df[['vulnerability_score', 'is_high_risk']].head())

Colonnes disponibles : ['source', 'report_id', 'age', 'sex', 'weight_kg', 'n_drugs', 'suspect_drugs', 'concomitant_drugs', 'all_drugs', 'avg_treatment_duration_days', 'n_symptoms', 'symptoms_text', 'narrative_text', 'outcome', 'serious', 'reporting_country', 'source_label', 'target_binary', 'target_risk', 'n_allergies', 'known_allergies', 'allergie_grave', 'n_chronic_diseases', 'chronic_diseases', 'n_concomitant_drugs', 'polypharmacie', 'score_risque_interaction', 'immunosuppression_anterieure', 'historique_traitements', 'risk_score_simulated', 'bmi', 'height_cm', 'region', 'substance_type', 'reaction_meddra', 'system_organ_class', 'has_immunosuppression', 'vaccine_or_drug', 'allergies_avec_reactions', 'classes_allergenes', 'severite_allergie_max', 'n_traitements_anterieurs', 'chirurgie_recente', 'dialyse', 'medicaments_concomitants', 'classes_therapeutiques', 'n_medicaments_risque_eleve', 'medicaments_risque_eleve', 'polymedicaments_haut_risque', 'narrative_words', 'severity_label']
✅

In [4]:
# Création d'identifiants uniques pour les entités
# Cela permettra de créer des liens (edges) entre Patients et Traitements
df['patient_id'] = range(len(df))
df['treatment_id'] = pd.factorize(df['vaccine_or_drug'])[0]

# Création d'une colonne de sévérité numérique simplifiée
df['severity_score'] = pd.factorize(df['severity_label'])[0]

print("✅ Identifiants pour Graph ML et scores de sévérité préparés.")

✅ Identifiants pour Graph ML et scores de sévérité préparés.


In [5]:
# Liste des colonnes que nous voulons garder pour l'entraînement
# On mélange données biomédicales classiques et tes nouveaux scores
cols_to_keep = [
    'age', 'sex', 'weight_kg', 'bmi', 'n_drugs', 'n_symptoms', 
    'n_allergies', 'n_chronic_diseases', 'score_risque_interaction', 
    'vulnerability_score', 'is_high_risk', 'severity_score',
    'target_binary', 'target_risk', 'symptoms_text' # On garde le texte pour BERT plus tard
]

# On filtre le dataframe (en vérifiant que la colonne existe)
df_final = df[[c for c in cols_to_keep if c in df.columns]]

print(f"✅ Sélection terminée : {df_final.shape[1]} colonnes conservées sur les 51 initiales.")

✅ Sélection terminée : 15 colonnes conservées sur les 51 initiales.


In [6]:
import os

# 1. Définition du chemin vers le dossier 'final'
FINAL_PATH = "../data/final/"

# 2. Création du dossier 'final' s'il n'existe pas
os.makedirs(FINAL_PATH, exist_ok=True)

# 3. Nouveau nom unique pour éviter les erreurs de doublons
# Ce fichier sera la source unique pour ton Notebook 04 (Modeling)
output_file = os.path.join(FINAL_PATH, "dataset_for_modeling.csv")

# 4. Sauvegarde effective de ton travail de Feature Engineering
df_final.to_csv(output_file, index=False)

print(f"🚀 Notebook 03 TERMINÉ !")
print(f"📍 Ton dataset d'entraînement unique est ici : {output_file}")
print(f"✅ Nouveau nom enregistré : dataset_for_modeling.csv")

🚀 Notebook 03 TERMINÉ !
📍 Ton dataset d'entraînement unique est ici : ../data/final/dataset_for_modeling.csv
✅ Nouveau nom enregistré : dataset_for_modeling.csv
